## SEED-IV without domain adaption 

In [1]:
"""
SEED-IV 4-Class Inductive Pipeline (Exact SEED Architecture - No Domain Adaptation)
===================================================================================
Specifications:
  1. Auto-Discovery Loader: Seamlessly locates session folders (1, 2, 3) under /kaggle/input.
  2. Exact SEED Architecture: SpatialSTMAE + Input Embed + MSCTimesNet (Top-K=2).
  3. No Domain Adaptation: Direct feedforward test inference under model.eval().
  4. Pretraining Schedule: 20 epochs per fold with GPU Mask Bank & stride=10 (~40s/fold).
  5. Multi-Session Isolated Consensus: Soft-voting across 72 trials (24 trials x 3 sessions).
  6. Row-Wise Reporting: Labels, per-channel metrics, and grand summary printed as table rows.
"""

from collections import defaultdict
import math
import os
import random
import re
import time
import warnings

import numpy as np
import pandas as pd
import scipy.io as sio
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_recall_fscore_support
from sklearn.model_selection import GroupShuffleSplit
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# =====================================================================
# 1. CONFIGURATION & DATASET PATHS
# =====================================================================
DATA_SEARCH_PATHS = [
    "/kaggle/input/datasets/phhasian0710/seed-iv/eeg_feature_smooth",
    "/kaggle/input/seed-iv/eeg_feature_smooth",
    "/kaggle/input/seed-iv-dataset/eeg_feature_smooth",
    "/kaggle/input/datasets/yunzinan/seed-iv-preprocessed/eeg_feature_smooth",
    "/kaggle/input/seed-iv-preprocessed/eeg_feature_smooth",
    ".",
]

CACHE_FILE = "/kaggle/working/seed_iv_phhasian_10dim_cache.npz"
OUTPUT_DIR = "/kaggle/working/seed_iv_seed_arch_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for old in [
    "/kaggle/working/seed_iv_40s_de_psd_cache.npz",
    "/kaggle/working/seed_iv_real_dataset_cache.npz",
]:
  if os.path.exists(old):
    try:
      os.remove(old)
    except OSError:
      pass

RANDOM_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CHANNELS = 62
NUM_CLASSES = 4  # 0: Neutral, 1: Sad, 2: Fear, 3: Happy
RAW_DIM = 10     # 5 log-PSD + 5 DE
WINDOW_LENGTH = 10
STRIDE = 1
TRIALS_PER_SESSION = 24

# STMAE Pretraining Parameters (50 Epochs, Accelerated)
FOLD_PRETRAIN_EPOCHS = 50
PRETRAIN_BATCH_SIZE = 256
PRETRAIN_LR = 1e-3
PRETRAIN_STRIDE = 10

# Supervised Fine-Tuning Parameters
CLASSIFIER_EPOCHS = 25
CLASSIFIER_BATCH_SIZE = 128
CLASSIFIER_LR = 1e-3
ENCODER_LR_SCALE = 0.1
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP = 1.0
LABEL_SMOOTHING = 0.05
CHANNEL_DROPOUT_RATE = 0.1

SEED_IV_TRIAL_LABELS = {
    1: [1, 2, 3, 0, 2, 0, 0, 1, 0, 1, 2, 1, 1, 1, 2, 3, 2, 2, 3, 3, 0, 3, 0, 3],
    2: [2, 1, 3, 0, 0, 2, 0, 2, 3, 3, 2, 3, 2, 0, 1, 1, 2, 1, 0, 3, 0, 1, 3, 1],
    3: [1, 2, 2, 1, 3, 3, 3, 1, 1, 2, 1, 0, 2, 3, 3, 0, 2, 3, 0, 0, 2, 0, 1, 0],
}

CHANNEL_NAMES = [
    "FP1", "FPZ", "FP2", "AF3", "AF4", "F7", "F5", "F3", "F1", "FZ",
    "F2", "F4", "F6", "F8", "FT7", "FC5", "FC3", "FC1", "FCZ", "FC2",
    "FC4", "FC6", "FT8", "T7", "C5", "C3", "C1", "CZ", "C2", "C4",
    "C6", "T8", "TP7", "CP5", "CP3", "CP1", "CPZ", "CP2", "CP4", "CP6",
    "TP8", "P7", "P5", "P3", "P1", "PZ", "P2", "P4", "P6", "P8",
    "PO7", "PO5", "PO3", "POZ", "PO4", "PO6", "PO8", "CB1", "O1", "OZ",
    "O2", "CB2",
]


def seed_everything(seed=RANDOM_SEED):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)


# =====================================================================
# 2. SCALP TOPOLOGY & 10 PHYSICAL REGIONS (PHY)
# =====================================================================
def build_scalp_coords():
  rows = [
      (["FP1", "FPZ", "FP2"], 0.95),
      (["AF3", "AF4"], 0.80),
      (["F7", "F5", "F3", "F1", "FZ", "F2", "F4", "F6", "F8"], 0.62),
      (["FT7", "FC5", "FC3", "FC1", "FCZ", "FC2", "FC4", "FC6", "FT8"], 0.42),
      (["T7", "C5", "C3", "C1", "CZ", "C2", "C4", "C6", "T8"], 0.20),
      (["TP7", "CP5", "CP3", "CP1", "CPZ", "CP2", "CP4", "CP6", "TP8"], -0.02),
      (["P7", "P5", "P3", "P1", "PZ", "P2", "P4", "P6", "P8"], -0.25),
      (["PO7", "PO5", "PO3", "POZ", "PO4", "PO6", "PO8"], -0.50),
      (["CB1", "O1", "OZ", "O2", "CB2"], -0.72),
  ]
  coords = {}
  for names, y in rows:
    n = len(names)
    xs = np.linspace(-1.0, 1.0, n) if n > 1 else np.array([0.0])
    for name, x in zip(names, xs):
      coords[name] = (float(x), float(y))
  return np.array([coords[c] for c in CHANNEL_NAMES], dtype=np.float32)


def build_phy_10_regions():
  coords = build_scalp_coords()
  regions = defaultdict(list)
  for idx, (x, y) in enumerate(coords):
    if y >= 0.70:
      reg = "prefrontal"
    elif y >= 0.35 and abs(x) <= 0.45:
      reg = "frontal_mid"
    elif y >= 0.35 and x < -0.45:
      reg = "frontal_left"
    elif y >= 0.35 and x > 0.45:
      reg = "frontal_right"
    elif -0.15 <= y < 0.35 and abs(x) <= 0.45:
      reg = "central"
    elif -0.15 <= y < 0.35 and x < -0.45:
      reg = "temporal_left"
    elif -0.15 <= y < 0.35 and x > 0.45:
      reg = "temporal_right"
    elif -0.55 <= y < -0.15 and x <= 0.0:
      reg = "parietal_left"
    elif -0.55 <= y < -0.15 and x > 0.0:
      reg = "parietal_right"
    else:
      reg = "occipital"
    regions[reg].append(idx)
  return {k: np.array(v, dtype=np.int64) for k, v in regions.items()}


SCALP_COORDS = build_scalp_coords()
PHY_REGIONS = build_phy_10_regions()


def precompute_mask_bank(num_masks=1024, device=DEVICE):
  masks = torch.zeros(num_masks, NUM_CHANNELS, dtype=torch.bool, device=device)
  region_keys = list(PHY_REGIONS.keys())
  for b in range(num_masks):
    if random.random() < 0.60:
      k = random.choice([3, 4])
      for rk in random.sample(region_keys, k):
        masks[b, torch.tensor(PHY_REGIONS[rk], device=device)] = True
    else:
      for _, ch_indices in PHY_REGIONS.items():
        n_ch = len(ch_indices)
        if n_ch <= 1:
          continue
        perm = torch.randperm(n_ch, device=device)
        mask_count = max(1, n_ch - 2)
        masks[b, torch.tensor(ch_indices[perm[:mask_count].cpu().numpy()], device=device)] = True
  return masks


MASK_BANK = precompute_mask_bank(1024, DEVICE)


def sample_phy_mask_fast(batch_size, seq_len=WINDOW_LENGTH):
  idx = torch.randint(0, len(MASK_BANK), (batch_size,), device=DEVICE)
  base_mask = MASK_BANK[idx]
  return base_mask.unsqueeze(1).expand(-1, seq_len, -1).reshape(-1, NUM_CHANNELS)


# =====================================================================
# 3. ROBUST SEED-IV FEATURE INGESTION & AUTO-DISCOVERY
# =====================================================================
def find_seed_iv_dir():
  for p in DATA_SEARCH_PATHS:
    if os.path.isdir(p):
      # Checks if directory contains session subfolders (1, 2, 3)
      if any(os.path.isdir(os.path.join(p, str(s))) for s in [1, 2, 3]):
        return p
  # Recursive search under /kaggle/input if default paths shift
  if os.path.isdir("/kaggle/input"):
    for root, dirs, _ in os.walk("/kaggle/input"):
      if "eeg_feature_smooth" in root and any(s in dirs for s in ["1", "2", "3"]):
        return root
  raise FileNotFoundError("Could not find SEED-IV eeg_feature_smooth directory across /kaggle/input.")


def load_seed_iv_features_10dim():
  if os.path.exists(CACHE_FILE):
    print(f"  [+] Loading cached 10-D dataset from: {CACHE_FILE}")
    d = np.load(CACHE_FILE)
    return d["X"], d["y"], d["subs"], d["sess"], d["tri"]

  data_dir = find_seed_iv_dir()
  print(f"  [+] Ingesting SEED-IV .mat archives from: {data_dir}")
  Xs, ys, subs, sess, tris = [], [], [], [], []

  for session in sorted(SEED_IV_TRIAL_LABELS.keys()):
    sdir = os.path.join(data_dir, str(session))
    if not os.path.isdir(sdir):
      continue
    labels = SEED_IV_TRIAL_LABELS[session]
    for fname in sorted(os.listdir(sdir)):
      if not fname.endswith(".mat"):
        continue
      subject = int(fname.split("_")[0])
      mat = sio.loadmat(os.path.join(sdir, fname))

      for t_num in range(1, TRIALS_PER_SESSION + 1):
        dk = f"de_LDS{t_num}" if f"de_LDS{t_num}" in mat else f"de_movingAve{t_num}"
        pk = f"psd_LDS{t_num}" if f"psd_LDS{t_num}" in mat else f"psd_movingAve{t_num}"

        if dk not in mat:
          continue

        de = mat[dk].astype(np.float32)
        if de.shape[0] == NUM_CHANNELS:
          de = np.transpose(de, (1, 0, 2))

        if pk in mat:
          psd = mat[pk].astype(np.float32)
          if psd.shape[0] == NUM_CHANNELS:
            psd = np.transpose(psd, (1, 0, 2))
          psd = np.log(np.maximum(psd, 1e-10))
          feat = np.concatenate([psd, de], axis=2)
        else:
          feat = de

        n = feat.shape[0]
        Xs.append(feat)
        ys.append(np.full(n, labels[t_num - 1], dtype=np.int64))
        subs.append(np.full(n, subject, dtype=np.int64))
        sess.append(np.full(n, session, dtype=np.int64))
        tris.append(np.full(n, t_num, dtype=np.int64))

  if len(Xs) == 0:
    raise RuntimeError(f"Found directory {data_dir}, but parsed 0 valid trials.")

  X = np.concatenate(Xs, axis=0)
  y = np.concatenate(ys)
  subs = np.concatenate(subs)
  sess = np.concatenate(sess)
  tri = np.concatenate(tris)

  print(f"  [+] Ingestion Complete: X={X.shape} | Labels={np.bincount(y)} | Subjects={np.unique(subs)}")
  np.savez_compressed(CACHE_FILE, X=X, y=y, subs=subs, sess=sess, tri=tri)
  return X, y, subs, sess, tri


def normalize_subject_session_independent(x, subs, sess):
  xn = x.copy()
  keys = subs * 100 + sess
  for k in np.unique(keys):
    m = keys == k
    blk = xn[m]
    mu = blk.mean(axis=0, keepdims=True)
    sd = blk.std(axis=0, keepdims=True) + 1e-6
    xn[m] = (blk - mu) / sd
  return xn


def build_sequence_framing(y, subs, sess, tri, seq_len=WINDOW_LENGTH, stride=STRIDE):
  keys = subs * 1000000 + sess * 10000 + tri
  seqs, labs, s_sub, s_ses, s_tri = [], [], [], [], []
  order = np.argsort(keys, kind="stable")
  for k in np.unique(keys):
    rows = order[keys[order] == k]
    if rows.shape[0] < seq_len:
      continue
    for start in range(0, rows.shape[0] - seq_len + 1, stride):
      win = rows[start : start + seq_len]
      seqs.append(win)
      labs.append(y[win[0]])
      s_sub.append(subs[win[0]])
      s_ses.append(sess[win[0]])
      s_tri.append(tri[win[0]])
  return (
      np.asarray(seqs, dtype=np.int64),
      np.asarray(labs, dtype=np.int64),
      np.asarray(s_sub, dtype=np.int64),
      np.asarray(s_ses, dtype=np.int64),
      np.asarray(s_tri, dtype=np.int64),
  )


# =====================================================================
# 4. NEUROSCIENCE-INFORMED SPATIAL STMAE (SEED ARCHITECTURE)
# =====================================================================
class ScalpPositionalEncoding(nn.Module):

  def __init__(self, coords, d_model):
    super().__init__()
    self.register_buffer("coords", torch.tensor(coords, dtype=torch.float32))
    self.mlp = nn.Sequential(
        nn.Linear(2, d_model), nn.GELU(), nn.Linear(d_model, d_model)
    )

  def forward(self, x):
    return x + self.mlp(self.coords).unsqueeze(0)


class SpatialSTMAE(nn.Module):

  def __init__(self, in_dim=RAW_DIM, d_model=64, latent_dim=32):
    super().__init__()
    self.latent_dim = latent_dim
    self.proj = nn.Linear(in_dim, d_model)
    self.pos_enc = ScalpPositionalEncoding(SCALP_COORDS, d_model)
    self.mask_token = nn.Parameter(torch.zeros(1, 1, d_model))
    nn.init.normal_(self.mask_token, std=0.02)

    enc_layer = nn.TransformerEncoderLayer(
        d_model=d_model,
        nhead=4,
        dim_feedforward=d_model * 2,
        dropout=0.1,
        batch_first=True,
        norm_first=True,
        activation="gelu",
    )
    self.encoder = nn.TransformerEncoder(enc_layer, num_layers=2)
    self.to_latent = nn.Sequential(
        nn.Linear(d_model, latent_dim), nn.LayerNorm(latent_dim)
    )
    self.decoder = nn.Sequential(
        nn.Linear(latent_dim, d_model), nn.GELU(), nn.Linear(d_model, in_dim)
    )

  def encode(self, x, mask=None):
    h = self.proj(x)
    if mask is not None:
      h = torch.where(mask.unsqueeze(-1), self.mask_token.expand_as(h), h)
    h = self.pos_enc(h)
    h = self.encoder(h)
    return self.to_latent(h)

  def forward_pretrain(self, x, mask):
    z = self.encode(x, mask)
    return self.decoder(z)

  def extract_latents(self, x, mask=None):
    B, T, C, F_dim = x.shape
    flat = x.reshape(B * T, C, F_dim)
    mask_flat = mask.expand(B * T, C) if mask is not None else None
    z = self.encode(flat, mask=mask_flat)
    return z.reshape(B, T, C * self.latent_dim)


def pretrain_stmae_fold_internal_amp_fast(
    x_pt,
    pt_seq_indices,
    train_rows_pt,
    in_dim=RAW_DIM,
    fold_name="Fold",
    epochs=FOLD_PRETRAIN_EPOCHS,
):
  stmae = SpatialSTMAE(in_dim=in_dim).to(DEVICE)
  opt = torch.optim.AdamW(stmae.parameters(), lr=PRETRAIN_LR, weight_decay=1e-4)
  sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
  scaler = torch.amp.GradScaler("cuda")

  tr_seqs = pt_seq_indices[train_rows_pt]
  n_seq = len(tr_seqs)
  stmae.train()

  print(f"    [+] Fast Pretraining STMAE on 14 Train Subjects ({epochs} epochs, AMP)...")
  for ep in range(1, epochs + 1):
    perm = torch.randperm(n_seq, device=DEVICE)
    tot_loss, steps = 0.0, 0
    for i in range(0, n_seq, PRETRAIN_BATCH_SIZE):
      b_idx = perm[i : i + PRETRAIN_BATCH_SIZE]
      xb = x_pt[tr_seqs[b_idx]]
      xb_flat = xb.reshape(-1, NUM_CHANNELS, in_dim)

      mask = sample_phy_mask_fast(xb.size(0), WINDOW_LENGTH)

      opt.zero_grad()
      with torch.amp.autocast("cuda"):
        recon = stmae.forward_pretrain(xb_flat, mask)
        loss = F.mse_loss(recon[mask], xb_flat[mask])

      scaler.scale(loss).backward()
      scaler.unscale_(opt)
      nn.utils.clip_grad_norm_(stmae.parameters(), GRADIENT_CLIP)
      scaler.step(opt)
      scaler.update()

      tot_loss += loss.item()
      steps += 1

    sched.step()
    if ep == 1 or ep % 5 == 0 or ep == epochs:
      avg_mse = tot_loss / max(steps, 1)
      print(f"      [{fold_name} - STMAE Fast] Epoch {ep:02d}/{epochs:02d} | Recon MSE: {avg_mse:.5f}")

  return stmae


# =====================================================================
# 5. TEMPORAL MSC-TIMESNET & CLASSIFIER (SEED ARCHITECTURE)
# =====================================================================
class MultiScaleConvBlock(nn.Module):

  def __init__(self, channels):
    super().__init__()
    mid = channels // 4
    self.b1 = nn.Conv2d(channels, mid, kernel_size=1, padding=0)
    self.b3 = nn.Conv2d(channels, mid, kernel_size=3, padding=1)
    self.b5 = nn.Conv2d(channels, mid, kernel_size=5, padding=2)
    self.bp = nn.Sequential(
        nn.AvgPool2d(kernel_size=3, stride=1, padding=1),
        nn.Conv2d(channels, mid, kernel_size=1),
    )
    self.fuse = nn.Conv2d(mid * 4, channels, kernel_size=1)
    self.bn = nn.BatchNorm2d(channels)
    self.act = nn.GELU()

  def forward(self, x):
    res = x
    out = torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bp(x)], dim=1)
    out = self.bn(self.fuse(out))
    return self.act(out + res)


class MSCTimesNet(nn.Module):

  def __init__(self, d_model=128, top_k=2):
    super().__init__()
    self.d_model = d_model
    self.top_k = top_k
    self.conv = MultiScaleConvBlock(d_model)
    self.norm = nn.LayerNorm(d_model)

  def forward(self, x):
    B, T, D = x.shape
    fft_energy = torch.abs(torch.fft.rfft(x, dim=1)).mean(dim=(0, 2))
    fft_energy[0] = 0.0

    _, topk_indices = torch.topk(fft_energy, self.top_k)
    periods = [
        max(2, int(round(T / idx.item()))) if idx.item() > 0 else T
        for idx in topk_indices
    ]

    period_outputs = []
    for p in periods:
      pad_len = (p - (T % p)) % p
      x_pad = F.pad(x, (0, 0, 0, pad_len)) if pad_len > 0 else x
      T_pad = x_pad.shape[1]

      x_2d = x_pad.reshape(B, T_pad // p, p, D).permute(0, 3, 1, 2)
      out_2d = self.conv(x_2d)
      out_1d = out_2d.permute(0, 2, 3, 1).reshape(B, T_pad, D)[:, :T, :]
      period_outputs.append(out_1d)

    agg = torch.stack(period_outputs, dim=0).mean(dim=0)
    return self.norm(x + agg)


class FullEmotionModel(nn.Module):

  def __init__(self, fold_stmae, num_classes=NUM_CLASSES):
    super().__init__()
    self.stmae = fold_stmae
    self.input_embed = nn.Sequential(
        nn.Linear(NUM_CHANNELS * 32, 128), nn.LayerNorm(128)
    )
    self.timesnet = MSCTimesNet(d_model=128, top_k=2)
    self.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(128, 64),
        nn.GELU(),
        nn.Linear(64, num_classes),
    )

  def forward(self, x, mask=None):
    h = self.stmae.extract_latents(x, mask=mask)
    h = self.input_embed(h)
    h = self.timesnet(h)
    return self.classifier(h.mean(dim=1))


# =====================================================================
# 6. SUPERVISED FINE-TUNING & DIRECT INFERENCE (NO DOMAIN ADAPTATION)
# =====================================================================
def fit_fold_classifier_amp(model, x_pt, seq_indices, train_rows, y_all, groups):
  gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_SEED)
  tr_local, va_local = next(
      gss.split(train_rows, y_all[train_rows], groups[train_rows])
  )

  tr_rows_pt = torch.tensor(train_rows[tr_local], dtype=torch.long, device=DEVICE)
  va_rows_pt = torch.tensor(train_rows[va_local], dtype=torch.long, device=DEVICE)
  labels_pt = torch.tensor(y_all, dtype=torch.long, device=DEVICE)

  opt = torch.optim.AdamW(
      [
          {"params": model.stmae.parameters(), "lr": CLASSIFIER_LR * ENCODER_LR_SCALE},
          {"params": model.input_embed.parameters(), "lr": CLASSIFIER_LR},
          {"params": model.timesnet.parameters(), "lr": CLASSIFIER_LR},
          {"params": model.classifier.parameters(), "lr": CLASSIFIER_LR},
      ],
      weight_decay=WEIGHT_DECAY,
  )

  sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CLASSIFIER_EPOCHS)
  criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
  scaler = torch.amp.GradScaler("cuda")

  best_f1, best_weights = -1.0, None

  for ep in range(1, CLASSIFIER_EPOCHS + 1):
    model.train()
    perm = torch.randperm(len(tr_rows_pt), device=DEVICE)
    for i in range(0, len(tr_rows_pt), CLASSIFIER_BATCH_SIZE):
      b_idx = tr_rows_pt[perm[i : i + CLASSIFIER_BATCH_SIZE]]
      xb = x_pt[seq_indices[b_idx]]
      yb = labels_pt[b_idx]

      if CHANNEL_DROPOUT_RATE > 0:
        keep = (torch.rand(xb.size(0), 1, NUM_CHANNELS, 1, device=DEVICE) > CHANNEL_DROPOUT_RATE).float()
        xb = xb * keep

      opt.zero_grad()
      with torch.amp.autocast("cuda"):
        logits = model(xb)
        loss = criterion(logits, yb)

      scaler.scale(loss).backward()
      scaler.unscale_(opt)
      nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
      scaler.step(opt)
      scaler.update()

    sched.step()

    if ep % 3 == 0 or ep == CLASSIFIER_EPOCHS:
      model.eval()
      with torch.no_grad(), torch.amp.autocast("cuda"):
        xb_va = x_pt[seq_indices[va_rows_pt]]
        preds_va = model(xb_va).argmax(dim=1).cpu().numpy()
        f1 = f1_score(
            y_all[train_rows[va_local]],
            preds_va,
            average="macro",
            zero_division=0,
        )
        if f1 > best_f1:
          best_f1 = f1
          best_weights = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

  if best_weights is not None:
    model.load_state_dict(best_weights)
  return model


@torch.no_grad()
def evaluate_test_subject_full_metrics(model, x_pt, seq_indices, test_rows, y_all, s_ses, s_tri):
  model.eval()
  te_rows_pt = torch.tensor(test_rows, dtype=torch.long, device=DEVICE)
  probs = []

  for i in range(0, len(test_rows), CLASSIFIER_BATCH_SIZE):
    batch = x_pt[seq_indices[te_rows_pt[i : i + CLASSIFIER_BATCH_SIZE]]]
    with torch.amp.autocast("cuda"):
      logits = model(batch)
      probs.append(F.softmax(logits, dim=1).cpu().numpy())

  probs = np.concatenate(probs, axis=0)
  y_true = y_all[test_rows]
  y_pred = probs.argmax(axis=1)

  w_acc = accuracy_score(y_true, y_pred)
  w_bacc = balanced_accuracy_score(y_true, y_pred)
  w_macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
  w_weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

  # Multi-Session Isolated Consensus: 24 trials x 3 sessions = Exactly 72 Trials
  unique_trial_keys = s_ses[test_rows] * 100 + s_tri[test_rows]
  yt_trial, yp_trial = [], []

  for t_k in np.unique(unique_trial_keys):
    idx = np.where(unique_trial_keys == t_k)[0]
    yt_trial.append(y_true[idx[0]])
    yp_trial.append(probs[idx].mean(axis=0).argmax())

  yt_trial, yp_trial = np.array(yt_trial), np.array(yp_trial)
  t_acc = accuracy_score(yt_trial, yp_trial)
  t_bacc = balanced_accuracy_score(yt_trial, yp_trial)
  t_macro_f1 = f1_score(yt_trial, yp_trial, average="macro", zero_division=0)
  t_weighted_f1 = f1_score(yt_trial, yp_trial, average="weighted", zero_division=0)

  return {
      "win_acc": w_acc,
      "win_bacc": w_bacc,
      "win_macro_f1": w_macro_f1,
      "win_weighted_f1": w_weighted_f1,
      "trial_acc": t_acc,
      "trial_bacc": t_bacc,
      "trial_macro_f1": t_macro_f1,
      "trial_weighted_f1": t_weighted_f1,
      "num_windows": len(test_rows),
      "num_trials": len(yt_trial),
  }


# =====================================================================
# 7. ROW-WISE LABEL & CHANNEL EVALUATORS
# =====================================================================
@torch.no_grad()
def evaluate_row_wise_breakdown(model, x_pt, last_test_indices, y_test, top_k=10):
  model.eval()
  n_samples = len(last_test_indices)
  num_channels = len(CHANNEL_NAMES)
  emotion_names = ["Neutral", "Sad", "Fear", "Happy"]

  all_probs = []
  for i in range(0, n_samples, CLASSIFIER_BATCH_SIZE):
    xb = x_pt[last_test_indices[i : i + CLASSIFIER_BATCH_SIZE]]
    with torch.amp.autocast("cuda"):
      logits = model(xb)
      all_probs.append(F.softmax(logits, dim=1).cpu().numpy())
  all_probs = np.concatenate(all_probs, axis=0)
  preds = all_probs.argmax(axis=1)

  # --- TABLE A: PER-EMOTION METRICS (LABELS ROW-WISE) ---
  prec, rec, f1, supp = precision_recall_fscore_support(y_test, preds, labels=[0, 1, 2, 3], zero_division=0)
  label_records = []
  for c_idx, e_name in enumerate(emotion_names):
    c_mask = y_test == c_idx
    c_acc = (preds[c_mask] == c_idx).sum() / max(c_mask.sum(), 1)
    label_records.append({
        "Emotion Label": e_name,
        "Class Accuracy": c_acc,
        "Precision": prec[c_idx],
        "Recall / Sensitivity": rec[c_idx],
        "F1-Score": f1[c_idx],
        "Support (Windows)": supp[c_idx],
    })
  df_labels = pd.DataFrame(label_records)

  print("\n" + "=" * 95)
  print("PER-EMOTION CLASSIFICATION METRICS (LABELS ROW-WISE):")
  print("=" * 95)
  print(df_labels.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

  # Standalone channel perturbation
  def predict_channel_masked(mask_1d):
    probs_list = []
    mask_2d = mask_1d.unsqueeze(0).to(DEVICE)
    for i in range(0, n_samples, CLASSIFIER_BATCH_SIZE):
      xb = x_pt[last_test_indices[i : i + CLASSIFIER_BATCH_SIZE]]
      with torch.amp.autocast("cuda"):
        logits = model(xb, mask=mask_2d)
        probs_list.append(F.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(probs_list, axis=0).argmax(axis=1)

  channel_records = []
  for ch_idx, ch_name in enumerate(CHANNEL_NAMES):
    single_mask = torch.ones(num_channels, dtype=torch.bool, device=DEVICE)
    single_mask[ch_idx] = False

    p_ch = predict_channel_masked(single_mask)
    acc = accuracy_score(y_test, p_ch)
    mac_f1 = f1_score(y_test, p_ch, average="macro", zero_division=0)

    sens = {}
    for c in range(NUM_CLASSES):
      m_c = y_test == c
      sens[emotion_names[c]] = (p_ch[m_c] == c).sum() / max(m_c.sum(), 1)

    channel_records.append({
        "Channel": ch_name,
        "Standalone Acc": acc,
        "Macro-F1": mac_f1,
        "Neutral Sens": sens["Neutral"],
        "Sad Sens": sens["Sad"],
        "Fear Sens": sens["Fear"],
        "Happy Sens": sens["Happy"],
    })
  df_channels = pd.DataFrame(channel_records)
  top_channels_df = df_channels.sort_values(by="Standalone Acc", ascending=False).reset_index(drop=True)

  # --- TABLE B: TOP-K STANDALONE CHANNELS (CHANNELS ROW-WISE) ---
  print("\n" + "=" * 95)
  print(f"TOP-{top_k} STANDALONE EEG CHANNELS (CHANNELS ROW-WISE):")
  print("=" * 95)
  print(top_channels_df.head(top_k).to_string(index=False, float_format=lambda v: f"{v:.4f}"))

  # --- TABLE C: TOP SPECIALIZED CHANNELS PER EMOTION (LABELS ROW-WISE) ---
  emotion_channel_records = []
  for c_idx, e_name in enumerate(emotion_names):
    col_name = f"{e_name} Sens"
    ranked = df_channels.sort_values(by=col_name, ascending=False).reset_index(drop=True)
    top_5 = ranked["Channel"].head(5).tolist()
    best_sens = ranked[col_name].iloc[0]
    best_acc = ranked["Standalone Acc"].iloc[0]
    emotion_channel_records.append({
        "Emotion Label": e_name,
        "Top-1 Channel": top_5[0],
        "Top-2 Channel": top_5[1],
        "Top-3 Channel": top_5[2],
        "Top-4 Channel": top_5[3],
        "Top-5 Channel": top_5[4],
        "Max Sensitivity": best_sens,
        "Standalone Acc": best_acc,
    })
  df_emotion_channels = pd.DataFrame(emotion_channel_records)

  print("\n" + "=" * 95)
  print("TOP SPECIALIZED CHANNELS PER EMOTION (LABELS ROW-WISE):")
  print("=" * 95)
  print(df_emotion_channels.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

  return df_labels, top_channels_df, df_emotion_channels


# =====================================================================
# 8. MAIN STRICT INDUCTIVE PIPELINE
# =====================================================================
def main():
  seed_everything(RANDOM_SEED)
  t0 = time.time()
  print("=" * 95)
  print("SEED-IV EXACT ARCHITECTURE PIPELINE (NO DOMAIN ADAPTATION)")
  print(f"Device: {DEVICE} | Channels: {NUM_CHANNELS} | Classes: {NUM_CLASSES} | Context: {WINDOW_LENGTH}s")
  print("=" * 95)

  # 1. Ingest Dual Features (log-PSD + DE)
  x_raw, y, subs, sess, tri = load_seed_iv_features_10dim()
  in_dim = x_raw.shape[-1]

  # 2. Independent Subject-Session Normalization
  x_norm = normalize_subject_session_independent(x_raw, subs, sess)
  x_pt = torch.tensor(x_norm, dtype=torch.float32, device=DEVICE)

  # 3. Dense Framing (Stride=1) for Fine-Tuning & Test Inference
  seq_idx, y_seq, s_sub, s_ses, s_tri = build_sequence_framing(
      y, subs, sess, tri, seq_len=WINDOW_LENGTH, stride=STRIDE
  )
  seq_idx_pt = torch.tensor(seq_idx, dtype=torch.long, device=DEVICE)
  groups = s_sub * 1000000 + s_ses * 10000 + s_tri
  unique_subs = np.unique(s_sub)

  # 4. Non-Overlapping Framing (Stride=10) Exclusively for Fast STMAE Pretraining
  pt_seq_idx, _, pt_s_sub, _, _ = build_sequence_framing(
      y, subs, sess, tri, seq_len=WINDOW_LENGTH, stride=PRETRAIN_STRIDE
  )
  pt_seq_idx_pt = torch.tensor(pt_seq_idx, dtype=torch.long, device=DEVICE)

  print(f"  [+] Dense Evaluation Windows (Stride=1): {len(seq_idx_pt)} across {len(unique_subs)} Subjects.")
  print(f"  [+] Fast Pretraining Windows (Stride=10): {len(pt_seq_idx_pt)} across {len(unique_subs)} Subjects.")

  loso_records = []
  last_trained_model = None
  last_test_indices = None
  last_test_labels = None

  print("\n" + "-" * 95)
  print("STARTING STRICT INDUCTIVE LOSO (15 SUBJECTS -- SEED-IV)")
  print("-" * 95)

  for target_sub in unique_subs:
    t_fold = time.time()
    tr_rows = np.where(s_sub != target_sub)[0]
    te_rows = np.where(s_sub == target_sub)[0]
    tr_rows_pt = np.where(pt_s_sub != target_sub)[0]

    fold_tag = f"Subject_{target_sub:02d}"
    print(f"\n>>> Fold: {fold_tag} (Train: 14 Subjects | Test: {fold_tag}) <<<")

    # Step 1: Pretrain STMAE strictly on 14 training subjects (20 epochs in ~40s)
    fold_stmae = pretrain_stmae_fold_internal_amp_fast(
        x_pt,
        pt_seq_idx_pt,
        tr_rows_pt,
        in_dim=in_dim,
        fold_name=fold_tag,
        epochs=FOLD_PRETRAIN_EPOCHS,
    )

    # Step 2: Supervised Fine-Tuning
    model = FullEmotionModel(fold_stmae, num_classes=NUM_CLASSES).to(DEVICE)
    model = fit_fold_classifier_amp(model, x_pt, seq_idx_pt, tr_rows, y_seq, groups)

    # Step 3: Pure Deterministic Inference under model.eval() (72 Trials)
    metrics = evaluate_test_subject_full_metrics(model, x_pt, seq_idx_pt, te_rows, y_seq, s_ses, s_tri)
    metrics["subject"] = fold_tag
    loso_records.append(metrics)

    last_trained_model = model
    last_test_indices = seq_idx_pt[te_rows]
    last_test_labels = y_seq[te_rows]

    print(
        f"  -> Finished {fold_tag} | WIN: acc={metrics['win_acc']:.4f} bacc={metrics['win_bacc']:.4f} macF1={metrics['win_macro_f1']:.4f} | "
        f"TRIAL: acc={metrics['trial_acc']:.4f} bacc={metrics['trial_bacc']:.4f} macF1={metrics['trial_macro_f1']:.4f} "
        f"({metrics['num_trials']}t / {(time.time() - t_fold) / 60.0:.2f} min)"
    )

  # Grand Summary CSV
  df_loso = pd.DataFrame(loso_records)
  df_loso.to_csv(os.path.join(OUTPUT_DIR, "results_strict_loso_seed_iv_exact_arch.csv"), index=False)

  # --- GRAND SUMMARY: HORIZONTAL ROW-WISE BENCHMARK FORMAT ---
  print("\n" + "=" * 95)
  print("STRICT INDUCTIVE LOSO GRAND SUMMARY: HORIZONTAL (ROW-WISE):")
  print("=" * 95)
  summary_row = pd.DataFrame([{
      "win_acc": df_loso["win_acc"].mean(),
      "win_bacc": df_loso["win_bacc"].mean(),
      "win_macro_f1": df_loso["win_macro_f1"].mean(),
      "win_weighted_f1": df_loso["win_weighted_f1"].mean(),
      "trial_acc": df_loso["trial_acc"].mean(),
      "trial_bacc": df_loso["trial_bacc"].mean(),
      "trial_macro_f1": df_loso["trial_macro_f1"].mean(),
      "trial_weighted_f1": df_loso["trial_weighted_f1"].mean(),
      "folds": len(df_loso),
  }])
  print(summary_row.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

  # --- GRAND SUMMARY: MEAN ± STD FORMAT ---
  print("\n" + "=" * 95)
  print("STRICT INDUCTIVE LOSO METRIC DISPERSION (MEAN ± STD):")
  print("=" * 95)
  metric_cols = [
      "win_acc", "win_bacc", "win_macro_f1", "win_weighted_f1",
      "trial_acc", "trial_bacc", "trial_macro_f1", "trial_weighted_f1",
  ]
  summary_df = pd.DataFrame({
      "Metric": metric_cols,
      "Mean": [df_loso[m].mean() for m in metric_cols],
      "Std": [df_loso[m].std() for m in metric_cols],
  })
  print(summary_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

  # Step 4: Row-Wise Label & Channel Sensitivity Evaluation
  if last_trained_model is not None:
    df_labels, df_top_ch, df_emo_ch = evaluate_row_wise_breakdown(
        model=last_trained_model,
        x_pt=x_pt,
        last_test_indices=last_test_indices,
        y_test=last_test_labels,
        top_k=10,
    )
    df_labels.to_csv(os.path.join(OUTPUT_DIR, "results_labels_row_wise.csv"), index=False)
    df_top_ch.to_csv(os.path.join(OUTPUT_DIR, "results_channels_row_wise.csv"), index=False)
    df_emo_ch.to_csv(os.path.join(OUTPUT_DIR, "results_emotion_channels_row_wise.csv"), index=False)

  print(f"\n[+] Full SEED-IV Pipeline Finished in {(time.time() - t0) / 60.0:.2f} minutes.")


if __name__ == "__main__":
  main()

SEED-IV EXACT ARCHITECTURE PIPELINE (NO DOMAIN ADAPTATION)
Device: cuda | Channels: 62 | Classes: 4 | Context: 10s
  [+] Ingesting SEED-IV .mat archives from: /kaggle/input/datasets/phhasian0710/seed-iv/eeg_feature_smooth
  [+] Ingestion Complete: X=(37575, 62, 10) | Labels=[10170 10245  9225  7935] | Subjects=[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]
  [+] Dense Evaluation Windows (Stride=1): 27855 across 15 Subjects.
  [+] Fast Pretraining Windows (Stride=10): 3255 across 15 Subjects.

-----------------------------------------------------------------------------------------------
STARTING STRICT INDUCTIVE LOSO (15 SUBJECTS -- SEED-IV)
-----------------------------------------------------------------------------------------------

>>> Fold: Subject_01 (Train: 14 Subjects | Test: Subject_01) <<<
    [+] Fast Pretraining STMAE on 14 Train Subjects (50 epochs, AMP)...
      [Subject_01 - STMAE Fast] Epoch 01/50 | Recon MSE: 0.79324
      [Subject_01 - STMAE Fast] Epoch 05/50 | Recon